In [1]:
import os
from google.colab import userdata


# Définissez vos informations de manière sécurisée
GITHUB_USER = "koubra-gaby"
#GITHUB_TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

# Configuration de l'URL avec authentification
REPO_URL = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Cloner le dépôt si ce n'est pas déjà fait
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("Dépôt cloné avec succès !")
else:
    print("Le dépôt existe déjà.")


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 381 (delta 14), reused 9 (delta 3), pack-reused 348 (from 1)
Receiving objects: 100% (381/381), 22.53 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (178/178), done.
Dépôt cloné avec succès !


## training fastext model for chadian arabic corpus

In [2]:
%%capture
!pip install gensim
!pip install nltk


In [3]:
## Importing libraries
from gensim.models import FastText
from gensim.test.utils import common_texts

In [4]:
import re
from nltk.tokenize import word_tokenize


def preparer_sentences_entrainer_fastext(fichier1, fichier2, file_to_save, lang="fr"):
    sentences = []

    for fichier in [fichier1, fichier2]:
        with open(fichier, "r", encoding="utf-8-sig") as f:
            for ligne in f:
                ligne = ligne.strip()

                if ligne:
                    # Remplacer les ponctuations par des espaces,
                    # sauf les apostrophes et les tirets
                    ligne = re.sub(
                        r"[^\w\s']",
                        " ",
                        ligne,
                        flags=re.UNICODE
                    )

                    # Tokenisation mot par mot
                    tokens = word_tokenize(
                        ligne,
                        preserve_line=True
                    )

                    if tokens:
                        sentences.append(tokens)

      ## training fastext
    model = FastText(
      sentences,
      vector_size=1024,
      window=5,
      min_count=1,
      epochs=10,
      workers=4,
      sg=1,
  )
    model.wv.save_word2vec_format(file_to_save)

    print("Model trained and savec successfully")

    return model,sentences

In [ ]:
## Loading the corpus then tokenizing it
data_path = "/content/Chadian_arabic-update/Data/full_dataset/"
model_arabic,sentences_arabic = preparer_sentences_entrainer_fastext(data_path + "arabe_tchad3.txt",data_path + "arabic_not_bible.txt", data_path + "arabic_embedding.vec")
sentences_arabic

In [6]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_arabic.wv.most_similar(
    "malak",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

Words similar to 'ana':
amalak : 0.8625
malakt : 0.8166
malakti : 0.7944
halak : 0.7893
mursaalak : 0.7748
za'alak : 0.7733
seefak : 0.7626
malja'ak : 0.755
Za'alak : 0.7437
azalak : 0.7394


In [ ]:
model_fench,sentences_french = preparer_sentences_entrainer_fastext(data_path + "V_francais_3.txt",data_path + "french_not_bible.txt", data_path + "fench_embedding.vec")
sentences_french

In [8]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_fench.wv.most_similar(
    "bonjour",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

Words similar to 'ana':
Séjour : 0.8575
jour : 0.8081
lourde : 0.8006
journellement : 0.7679
lourd : 0.7671
bonheur : 0.7642
jour-ci : 0.7463
jour-aujourd'hui-en : 0.7273
lumignon : 0.7078
bonbon : 0.7045


In [9]:
!git clone https://github.com/facebookresearch/MUSE.git

Cloning into 'MUSE'...
remote: Enumerating objects: 239, done.
remote: Total 239 (delta 0), reused 0 (delta 0), pack-reused 239 (from 1)
Receiving objects: 100% (239/239), 215.77 KiB | 30.82 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [21]:
%cd /content/MUSE

from pathlib import Path

path = Path("src/utils.py")
code = path.read_text()

code = code.replace(
    "inspect.getargspec(optim_fn.__init__)[0]",
    "inspect.getfullargspec(optim_fn.__init__).args"
)

path.write_text(code)

print("Modification appliquée.")

/content/MUSE
Modification appliquée.


In [ ]:
!python unsupervised.py \
    --src_lang arb \
    --tgt_lang fr \
    --src_emb /content/Chadian_arabic-update/Data/full_dataset/fench_embedding.vec \
    --tgt_emb /content/Chadian_arabic-update/Data/full_dataset/arabic_embedding.vec \
    --emb_dim 1024 \
    --n_refinement 5 \
    --dis_most_frequent 20000

In [30]:
%cd /content/MUSE

from pathlib import Path

path = Path("unsupervised.py")
code = path.read_text()

code = code.replace(
    "    evaluator.all_eval(to_log)",
    "    # evaluator.all_eval(to_log)"
)

path.write_text(code)
print("Appel à l'évaluation désactivé.")

/content/MUSE
Appel à l'évaluation désactivé.


In [ ]:
from pathlib import Path

path = Path("/content/MUSE/unsupervised.py")
code = path.read_text()

code = code.replace(
    "    trainer.save_best(to_log, VALIDATION_METRIC)",
    "    # trainer.save_best(to_log, VALIDATION_METRIC)"
)

path.write_text(code)
print("Sauvegarde conditionnée à la métrique désactivée.")

In [32]:
!grep -n "save_best\|trainer.save\|export" /content/MUSE/unsupervised.py

33:parser.add_argument("--export", type=str, default="txt", help="Export embeddings after training (txt / pth)")
91:assert params.export in ["", "txt", "pth"]
144:        trainer.save_best(to_log, VALIDATION_METRIC)
179:        trainer.save_best(to_log, VALIDATION_METRIC)
183:# export embeddings
184:if params.export:
186:    trainer.export()
